# Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

1. Tracking agent behavior with logging, analytics, and debugging.
2. Transforming prompts, tool selection, and output formatting.
3. Adding retries, fallbacks, and early termination logic.
4. Applying rate limits, guardrails, and PII detection.

In [7]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"]= os.getenv("GROQ_API_KEY")
model= init_chat_model("llama-3.3-70b-versatile",
                       model_provider= "groq"
                       )


### built-in middleware

1. summarization -> agent
2. human-in-the-loop
3. model call limit
4. to-do list
5. llm tool selector
and many more


### Summarization Middleware

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

1. Long-running conversations that exceed context windows.
2. Multi-turn dialogues with extensive history.
3. Applications where preserving full conversation context matters.

In [8]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage


### Messagebased summarization
agent = create_agent(
    model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model,
            trigger=("messages", 10),
            keep=("messages", 4)
        )
    ]
)

In [9]:
# run with thread id
config={"configurable": {"thread_id":"test-1"}}

In [4]:
# Alternative test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response = agent.invoke(
        {"messages": [HumanMessage(content=q)]},
        config
    )
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='f93bd651-30cc-49ab-90cf-10d6ba62eb71'), AIMessage(content='2 + 2 = 4', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f4b6b-6924-7ca1-87fb-cb5bf8238df6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 28, 'total_tokens': 36, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 21}})]}
Messages: 2
Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='f93bd651-30cc-49ab-90cf-10d6ba62eb71'), AIMessage(content='2 + 2 = 4', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f4b6b-6924-7ca1-87fb-cb5bf8238df6-0', tool_c

### Based on Token size

In [10]:
# token Size

from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver


@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


agent = create_agent(
    model,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model,
            trigger=("tokens", 550),
            keep=("tokens", 200)
        )
    ]
)

config= {"configurable":{"thread_id":"test-1"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ≈ 1 token

In [11]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{response['messages']}")

Paris: ~242 tokens, 6 messages
[HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\nFind hotels in Paris.\n\n## SUMMARY\nThe user is looking for hotels in Paris. The search results include three hotels: Grand Hotel (5-star, $350/night), City Inn (4-star, $180/night), and Budget Stay (3-star, $75/night). The search has been repeated multiple times with the same results.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nSelect a hotel from the search results or refine the search criteria to find more options.', additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='cbdf899f-1ed0-4e8c-8bfd-2ea699731bfa'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '9n4s6sdae', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 493, 'total_tokens': 508, 'completion_time': 0.016885507, 'completion_tokens_details': None,

### Based on fraction

In [12]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver


@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"


# LOW fraction for testing!
agent = create_agent(
    model,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model,
            trigger=("fraction", 0.005),  # 0.5% = ~640 tokens
            keep=("fraction", 0.002),     # 0.2% = ~256 tokens
        ),
    ],
)


config = {"configurable": {"thread_id": "test-1"}}


# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4


# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )

    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000  # gpt-4o-mini context

    print(
        f"{city}: ~{tokens} tokens "
        f"({fraction:.4%}), "
        f"{len(response['messages'])} msgs"
    )

    print(response["messages"])

Paris: ~57 tokens (0.0445%), 6 msgs
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='333bb90c-1566-446c-a541-8ed8964c4645'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '81c1p42z0', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 210, 'total_tokens': 225, 'completion_time': 0.045859483, 'completion_tokens_details': None, 'prompt_time': 0.010538439, 'prompt_tokens_details': None, 'queue_time': 0.161328459, 'total_time': 0.056397922}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f4b70-a33c-7f12-b22b-9f3c67fd3de7-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': '81c1p42z0', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metada